In [1]:
ls


01_DESI_DR1_DataPreparation.ipynb  final_triangle.pdf  TWEB_ASTRA/
17716780.zip                       LRG/                TWEB_ASTRA.ipynb
BGS/                               notebooks/
final_1d.pdf                       papers/


In [2]:
cd TWEB_ASTRA

/home/qarni/galaxy_evolution_project/TWEB_ASTRA


/home/qarni/miniforge3/envs/galaxy_env/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
ls


classification/        groups.tar.gz  probabilities/        raw.tar.gz
classification.tar.gz  pairs/         probabilities.tar.gz
groups/                pairs.tar.gz   raw/


In [6]:
###############################################################
# bgs T - W E B   R E C O N S T R U C T I O N
# Using ASTRA DESI DR1 raw catalogs (NGC1 + NGC2)
###############################################################

import numpy as np
import h5py
from astropy.table import Table
from scipy.ndimage import gaussian_filter
from numpy.fft import rfftn, irfftn, fftfreq
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


######################################################################
# 1. LOAD RAW ASTRA CATALOGS (NGC1 + NGC2)
######################################################################

raw1 = Table.read("raw/zone_NGC1.fits.gz")
raw2 = Table.read("raw/zone_NGC2.fits.gz")
raw = Table(np.hstack([raw1, raw2]))

print("Total objects loaded:", len(raw))


######################################################################
# 2. FILTER bgs DATA + RANDOMS
######################################################################

bgs_data = raw[(raw["TRACERTYPE"]=="BGS_DATA")]
bgs_rand = raw[(raw["TRACERTYPE"]=="BGS_RAND")]

print("BGS data:", len(bgs_data))
print("BGS random:", len(bgs_rand))


######################################################################
# 3. EXTRACT COORDINATES
######################################################################

gal = np.vstack([
    bgs_data["XCART"], 
    bgs_data["YCART"], 
    bgs_data["ZCART"]
]).T.astype(np.float32)

ran = np.vstack([
    bgs_rand["XCART"], 
    bgs_rand["YCART"], 
    bgs_rand["ZCART"]
]).T.astype(np.float32)


######################################################################
# 4. AUTOMATIC SMART CUBE SELECTION
#    Ensures both data + randoms exist inside cube
######################################################################

# Coarse grid to find overlap
def find_overlap_cube(gal, ran, N=20):
    mins = np.minimum(gal.min(0), ran.min(0))
    maxs = np.maximum(gal.max(0), ran.max(0))
    span = maxs - mins

    def coarse(points):
        idx = np.floor((points - mins)/span * N).astype(int)
        idx = np.clip(idx, 0, N-1)
        G = np.zeros((N,N,N), np.int32)
        for i,j,k in idx:
            G[i,j,k]+=1
        return G

    Gg = coarse(gal)
    Gr = coarse(ran)

    mask = (Gg>500) & (Gr>500)

    if not np.any(mask):
        raise RuntimeError("NO voxel contains both galaxies and randoms.")

    vox = np.unravel_index(np.argmax(Gg*mask), mask.shape)
    return vox, mins, span

(ix,iy,iz), mins, span = find_overlap_cube(gal, ran)

print("Selected voxel:", ix, iy, iz)

# define cube around center of voxel
voxel_min = mins + span * np.array([ix,iy,iz])/20
voxel_max = mins + span * np.array([ix+1,iy+1,iz+1])/20
center = 0.5*(voxel_min + voxel_max)

cube_size = 1000.0
half = cube_size/2
cube_min = center - half
cube_max = center + half


######################################################################
# 5. CROP DATA TO THIS CUBE
######################################################################

def crop(p, cmin, cmax):
    m = (
        (p[:,0]>=cmin[0]) & (p[:,0]<=cmax[0]) &
        (p[:,1]>=cmin[1]) & (p[:,1]<=cmax[1]) &
        (p[:,2]>=cmin[2]) & (p[:,2]<=cmax[2])
    )
    return p[m]

gal_c = crop(gal, cube_min, cube_max)
ran_c = crop(ran, cube_min, cube_max)

print("Cropped galaxies:", gal_c.shape)
print("Cropped randoms :", ran_c.shape)

if len(ran_c)==0:
    raise RuntimeError("Randoms zero — enlarge cube_size.")


######################################################################
# 6. REBOX INTO UNIFORM [0, L]^3
######################################################################

Lbox = 640.0
mins_c = np.minimum(gal_c.min(0), ran_c.min(0))
maxs_c = np.maximum(gal_c.max(0), ran_c.max(0))
span_c = maxs_c - mins_c

gal_r = (gal_c - mins_c)/span_c * Lbox
ran_r = (ran_c - mins_c)/span_c * Lbox


######################################################################
# 7. CIC GRID
######################################################################

Ngrid = 128
dx = Lbox/Ngrid

@njit(parallel=True)
def cic_grid(points, box, N):
    dx = box/N
    grid = np.zeros((N,N,N), np.float32)
    for p in prange(points.shape[0]):
        x,y,z = points[p]
        i=int(x/dx); j=int(y/dx); k=int(z/dx)
        if 0<=i<N and 0<=j<N and 0<=k<N:
            grid[i,j,k]+=1
    return grid

Ng = cic_grid(gal_r, Lbox, Ngrid)
Nr = cic_grid(ran_r, Lbox, Ngrid)


######################################################################
# 8. DENSITY CONTRAST
######################################################################

alpha = Ng.sum()/Nr.sum()
Nr *= alpha
delta = (Ng - Nr)/(Nr + 1e-6)
delta = np.nan_to_num(delta)


######################################################################
# 9. SMOOTHING
######################################################################

sigma = 6.0/dx
delta_s = gaussian_filter(delta, sigma=sigma)


######################################################################
# 10. TIDAL TENSOR VIA FFT
######################################################################

kx = fftfreq(Ngrid, d=dx)*2*np.pi
ky = fftfreq(Ngrid, d=dx)*2*np.pi
kz = fftfreq(Ngrid, d=dx)*2*np.pi
kx,ky,kz = np.meshgrid(kx,ky,kz[:Ngrid//2+1], indexing='ij')

k2 = kx*kx + ky*ky + kz*kz
k2[0,0,0] = 1e-6

phi_k = - rfftn(delta_s)/k2

Txx = irffnn(kx*kx*phi_k, delta_s.shape)
Tyy = irffnn(ky*ky*phi_k, delta_s.shape)
Tzz = irffnn(kz*kz*phi_k, delta_s.shape)
Txy = irffnn(kx*ky*phi_k, delta_s.shape)
Txz = irffnn(kx*kz*phi_k, delta_s.shape)
Tyz = irffnn(ky*kz*phi_k, delta_s.shape)


######################################################################
# 11. EIGENVALUES AND WEB-CLASS
######################################################################

lambda1 = np.zeros_like(delta_s)
lambda2 = np.zeros_like(delta_s)
lambda3 = np.zeros_like(delta_s)

@njit(parallel=True)
def eigs(Txx,Tyy,Tzz,Txy,Txz,Tyz,l1,l2,l3):
    N=Txx.shape[0]
    for i in prange(N):
        for j in range(N):
            for k in range(N):
                M = np.array([
                    [Txx[i,j,k],Txy[i,j,k],Txz[i,j,k]],
                    [Txy[i,j,k],Tyy[i,j,k],Tyz[i,j,k]],
                    [Txz[i,j,k],Tyz[i,j,k],Tzz[i,j,k]]
                ])
                w = np.linalg.eigvalsh(M)
                l1[i,j,k], l2[i,j,k], l3[i,j,k] = w[2], w[1], w[0]

eigs(Txx,Tyy,Tzz,Txy,Txz,Tyz, lambda1,lambda2,lambda3)

env = np.zeros_like(lambda1, np.int8)
env[(lambda1<0)] = 0
env[(lambda1>0)&(lambda2<0)] = 1
env[(lambda2>0)&(lambda3<0)] = 2
env[(lambda3>0)] = 3

print("Web types:", np.unique(env))


######################################################################
# 12. VISUALIZATION
######################################################################

colors = [(0,0,0), (0.2,0.4,1), (1,1,0.2), (1,0,0)]
cmap = mcolors.ListedColormap(colors)

z=Ngrid//2
sl=env[:,:,z]

plt.figure(figsize=(8,8))
plt.imshow(sl.T, origin="lower", cmap=cmap, vmin=0, vmax=3)
plt.title("bgs T-Web Classification")
plt.colorbar()
plt.show()


Total objects loaded: 50292748
BGS data: 0
BGS random: 0


ValueError: zero-size array to reduction operation minimum which has no identity